In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\RAHUL\AppData\Local\Temp\ipykernel_20028\2692255167.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")
jina_api_key = os.getenv("JINA_API_KEY")

### DATA LOADING

In [4]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

In [5]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()

print("=="*30)
print(f"Loaded {len(documents)} documents from {DATA_FILE_PATH}, each document has {len(documents[0].page_content)} characters.")
print(documents)

Loaded 1 documents from data\hr_policy.txt, each document has 2597 characters.
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees und

### SPLITING OUR DATA 

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks.")

Split into 9 chunks.


In [7]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


### EMBEDDINGS

In [8]:
embeddings_model = JinaEmbeddings(jina_api_key=jina_api_key, model_name="jina-embeddings-v5-text-small")
print("Embedding model initialized successfully.", "Model name:", embeddings_model.model_name)

Embedding model initialized successfully. Model name: jina-embeddings-v5-text-small


### STORE DATA IN VECTOR DB

In [9]:
vectore_store = FAISS.from_documents(chunks, embeddings_model)

print("Vector store created successfully with FAISS.", vectore_store.index.ntotal, "vectors in the index.")

Vector store created successfully with FAISS. 9 vectors in the index.


WE NEED TO STORE YOUR DATA

In [10]:
test_query = "How many sick leaves employess get"

## SIMILARITY SEARCH

top_matches = vectore_store.similarity_search(test_query, k=2)

print(f"Top {len(top_matches)} matches for the query '{test_query}':")
print("=="*30)
for i, match in enumerate(top_matches):
    print(f"Match {i+1}:")
    print(f"Content: {match.page_content}")
    print(f"Metadata: {match.metadata}")
    print("=="*30)

Top 2 matches for the query 'How many sick leaves employess get':
Match 1:
Content: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
Metadata: {'source': 'data\\hr_policy.txt'}
Match 2:
Content: 7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.
Metadata: {'source': 'data\\hr_policy.txt'}


TOOL

In [23]:
retriever = vectore_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 2}
)

def search_hr_policy(query: str) -> str:
    """
    Search the HR policy handbook for information about leave, work from home, probation, notice period, reimbursement, code of conduct, holidays  or exit process. Returns the most relevant information from the handbook.
    """
    matching_chunks = retriever.invoke(query)
    return "\n\n".join([chunk.page_content for chunk in matching_chunks])


### DATA RETRIVAL

In [30]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5 #creativity of the model
)

llm.model_name



'openai/gpt-oss-120b'

In [25]:
test_response = llm.invoke("Hey is learning RAG Hard? reply in a joke manner, in one line")

test_response.content

'Learning RAG is like teaching a cat to fetch—possible, but expect a lot of scratching, confused meows, and the occasional “I’m just here for the laser pointer!”'

AI AGENT

In [26]:
from langchain.agents import create_agent

hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
        You are a friendly HR assistant.
        Always use the search_hr_policy tool to look up
        facts before answering.
        If the answer isn't in the search results, say you don't know instead of guessing.
    """
)

print("HR Assistant created successfully.")

HR Assistant created successfully.


In [27]:
def ask_hr_assistant(query: str) -> str:
    """Send a query to the HR assistant and print the answer."""
    print("==" * 30)
    print(f"Query: {query}")

    response = hr_assistant.invoke({
        "messages": [
            {"role": "user", "content": query}
        ]
    })
    answer = response["messages"][-1].content
    print(f"Answer: {answer}")
    print("==" * 30)
    return answer

In [31]:
response = hr_assistant.invoke({
    "messages": [
        {"role": "user", "content": "How many sick leaves employess get"}
    ]
})

print(response["messages"][-1].content)

Employees are entitled to **10 paid sick days per year**. (A medical certificate is required for sick leave longer than two consecutive days.)
